In [1]:
!ls

chunk_embeddings  chunks_df.csv  data  embeddings.zip  qdrant


In [2]:
!pip install --upgrade langchain-openai
!pip install --upgrade langchain
!pip install --upgrade langchain-core
!pip install --upgrade langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.9/438.9 kB 11.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.7/367.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: openai
    Found existing installation: openai 1.70.0
    Uninstalling openai-1.70.0:
      Successfully uninstalled openai-1.70.0
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.23
    Uninstalling langsmith-0.3.23:
      Successfully uninstalled langsmith-0.3.23
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.50
    Uninstalling la

In [3]:
!pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 6.3 MB/s eta 0:00:00a 0:00:01


In [4]:
import pandas as pd
import numpy as np
import os
from qdrant_client.http import models
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import base64
from IPython.display import HTML

In [5]:
DATA_PATH = "data"
DOCS_PATH = os.path.join(DATA_PATH, "docs")
EMBEDDINGS_PATH = "chunk_embeddings"

In [6]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Vector db

In [7]:
from qdrant_client import QdrantClient


client = QdrantClient(path="qdrant")

In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-m3")

2025-06-30 14:18:27.826676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751293108.023499      75 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751293108.080992      75 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [12]:
collection_name = "freecad_docs"

query = "create a cylinder"
query_vec = embedding_model.encode([query])[0]
hits = client.query_points(
    collection_name=collection_name,
    query=query_vec,
    limit=3,
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
for point in hits.points:
    print("===================\n")
    print("Score: ", point.score)
    print("Source", point.payload['source'])
    print(point.payload['content'])


Score:  0.68448202601723
Source Scripts
Let's start with a small script that does a very small job, but display the power of this approach.

```

 Script functions

def my_box(name, len, wid, hei):
    """Create a box."""
    obj_b = DOC.addObject("Part::Box", name)
    obj_b.Length = len
    obj_b.Width = wid
    obj_b.Height = hei

    DOC.recompute()

    return obj_b

 objects definition

obj = my_box("test_cube", 5, 5, 5)

set_view()

```

Write above lines of code after  and press the green arrow in the Macro toolbar.

You will see some magic things, a new document is open named "Wiki_example" and you will see in the 3d view a Cube, like the one in the image below.

 Something more 

Not that amazing? Yes, but we have to start somewhere, we can do the same thing with a Cylinder, add these lines of code after the  function and before the line: .

```

def my_cyl(name, ang, rad, hei):
    """Create a Cylinder."""
    obj = DOC.addObject("Part::Cylinder", name)
    obj.Angle = ang


# LLM

In [ ]:
# from langchain_huggingface.llms import HuggingFacePipeline

In [9]:
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

qwen_model = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    qwen_model,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    qwen_model,
    trust_remote_code=True,
    device_map=device
)

text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024
)
llm = HuggingFacePipeline(pipeline=text_gen)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda


In [26]:
SYSTEM_PROMPT = (
"""
Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
"""
)

def format_prompt(user_message: str, context: str = None) -> str:
    '''
    Formats prompt for llm
    '''
    
    prompt = []
    prompt.append({
            "role": "system",
            "content": SYSTEM_PROMPT
    })


    if context:
        prompt.append({
            "role": "system",
            "content": "Context:\n\n" + context
        })

    prompt.append({
        "role": "user",
        "content": user_message
    })


    return tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True
    )

In [27]:
def get_context_from_hits(hits, max_chunks=3):
    """Extract and concatenate content from top hits."""
    docs = []
    for point in hits.points[:max_chunks]:
        docs.append(point.payload['content'])
    return "\n\n".join(docs)

def rag_query(query, client, embedding_model, collection_name, llm, top_k=3):
    query_vec = embedding_model.encode([query])[0]
    hits = client.query_points(
        collection_name=collection_name,
        query=query_vec,
        limit=top_k,
    )

    context = get_context_from_hits(hits, max_chunks=top_k)

    prompt = format_prompt(query, context)
    response = llm(prompt)
    return response


query = "Add wall with 200x1000 size"
response = rag_query(query, client, embedding_model, collection_name, llm)
print(response)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


<|im_start|>system

Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
<|im_end|>
<|im_start|>system
Context:

Walls can also have additions or subtractions. Additions are other objects whose shapes are joined in this Wall's shape, while subtractions are